### Blood Report Extraction - MinIO PDF → PostgreSQL

**Pipeline:**
```
MinIO (PDF)
  → Download temporarily (Both Tabula and Tesseract are local processing libraries,only know how to read files from machine's disk.)
  → Tabula  (digital PDFs — extracts tables)
  → Tesseract (scanned PDFs — OCR fallback)
  → Parse key markers (glucose, hemoglobin, cholesterol …)
  → Validate values
  → Store in PostgreSQL blood_reports table
```

**Duplicate prevention:** `(customer_id, report_date, lab_ref, lab_name)` unique constraint — re-running is always safe.

#### 1. Install dependencies

In [ ]:
!pip install minio psycopg2-binary python-dotenv tabula-py pytesseract pdf2image pillow pandas --quiet

# Tesseract binary must also be installed on your machine:
# Windows  → https://github.com/UB-Mannheim/tesseract/wiki  (add to PATH)
# Mac      → brew install tesseract
# Linux    → sudo apt install tesseract-ocr

In [1]:
!pip install pdfplumber --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 2. Imports & connections

In [2]:
import os
import re
import json
import tempfile
import psycopg2
import tabula
import pytesseract
import pandas as pd
from minio import Minio
from pdf2image import convert_from_path
from dotenv import load_dotenv
import pdfplumber

load_dotenv()



True

In [3]:
# ── clients ──────────────────────────────────────────────
def get_minio():
    return Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False
    )

def get_pg():
    return psycopg2.connect(
        host="localhost", port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

minio = get_minio()
pg    = get_pg()
print("✅ Connected")

✅ Connected


#### 3. Create `blood_reports` table (run once)

In [4]:
CREATE_TABLE = """
CREATE TABLE IF NOT EXISTS blood_reports (
    id                  SERIAL PRIMARY KEY,

    -- identity
    customer_id         TEXT NOT NULL,
    report_date         DATE,
    lab_name            TEXT,
    lab_ref             TEXT,

    -- key blood markers (NULL = not found in report)
    glucose             DECIMAL(6,2),   -- mg/dL  normal: 70–100
    hemoglobin          DECIMAL(5,2),   -- g/dL   normal: 12–17
    cholesterol_total   DECIMAL(6,2),   -- mg/dL  normal: <200
    cholesterol_hdl     DECIMAL(6,2),   -- mg/dL  normal: >40
    cholesterol_ldl     DECIMAL(6,2),   -- mg/dL  normal: <100
    triglycerides       DECIMAL(6,2),   -- mg/dL  normal: <150
    wbc                 DECIMAL(6,2),   -- x10³/µL normal: 4–11
    rbc                 DECIMAL(6,2),   -- x10⁶/µL normal: 4.5–5.5

    -- flexible storage for all other markers
    raw_data            JSONB,

    -- traceability
    minio_path          TEXT,
    extraction_method   TEXT,           -- tabula | tesseract
    is_valid            BOOLEAN DEFAULT TRUE,
    validation_notes    TEXT,
    indexed_at          TIMESTAMPTZ DEFAULT NOW(),

    -- prevent duplicate records
    UNIQUE (customer_id, report_date, lab_name, lab_ref)
);
"""

with pg.cursor() as cur:
    cur.execute(CREATE_TABLE)
pg.commit()
print("✅ Table ready")

✅ Table ready


#### 4. PDF text extraction
Tries **Tabula** first (fast, for digital PDFs). Falls back to **Tesseract** for scanned PDFs.

In [5]:

def extract_text_tabula(pdf_path):
    """Extract text using pdfplumber — no Java needed."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # extract tables
            for table in page.extract_tables():
                for row in table:
                    text += " ".join(str(c) for c in row if c) + "\n"
            # extract plain text too
            text += page.extract_text() or ""
    return text, "pdfplumber"


def extract_text_tesseract(pdf_path):
    """OCR fallback for scanned PDFs."""
    images = convert_from_path(pdf_path, dpi=300)
    text   = " ".join(pytesseract.image_to_string(img) for img in images)
    return text, "tesseract"


def extract_text(pdf_path):
    """Try Tabula first; fall back to Tesseract if no content found."""
    text, method = extract_text_tabula(pdf_path)
    if len(text.strip()) < 50:          # too little text → probably scanned
        text, method = extract_text_tesseract(pdf_path)
    return text, method

#### 5. Parse blood markers from extracted text

In [22]:
# Each entry: (field_name, regex_pattern)
MARKERS = [
    ("glucose",           r"glucose[:\s]+([\d.]+)"),
    ("hemoglobin",        r"h(?:a?e)?moglobin[:\s]+([\d.]+)"),
    ("cholesterol_total", r"(?:total\s+)?cholesterol[:\s]+([\d.]+)"),
    ("cholesterol_hdl",   r"hdl[:\s]+([\d.]+)"),
    ("cholesterol_ldl",   r"ldl[:\s]+([\d.]+)"),
    ("triglycerides",     r"triglycerides?[:\s]+([\d.]+)"),
    ("wbc",               r"(?:wbc|white\s+blood\s+cells?)[:\s]+([\d.]+)"),
    ("rbc",               r"(?:rbc|red\s+blood\s+cells?)[:\s]+([\d.]+)"),
]

def parse_markers(text):
    """Extract blood marker values from raw text using regex."""
    text_lower = text.lower()
    result = {}
    for field, pattern in MARKERS:
        match = re.search(pattern, text_lower)
        result[field] = float(match.group(1)) if match else None
    return result


import re

def parse_meta(text):
    """Extract report date, lab name, and lab reference from text."""

    date_match = re.search(
        r"report\s*date[:\s]+([\d]{1,2}\s+[A-Za-z]+\s+\d{4})",
        text, re.IGNORECASE
    )

    # fallback formats
    if not date_match:
        date_match = re.search(
            r"report\s*date[:\s]+([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})",
            text, re.IGNORECASE
        )


    lab_match = re.search(
        r"(?:lab(?:oratory)?|clinic|hospital)[:\s]+([A-Za-z0-9 ]+)",
        text, re.IGNORECASE
    )
    ref_match = re.search(
    r"(?:lab\s*ref(?:erence)?|ref(?:erence)?\s*no|report\s*id)\s*[:\-]?\s*\n?\s*([A-Za-z0-9\-\/]+)",
    text, re.IGNORECASE
    )

 

    report_date = date_match.group(1) if date_match else None
    lab_name    = lab_match.group(1).strip() if lab_match else "Unknown"
    lab_ref     = ref_match.group(1).strip() if ref_match else None

    return report_date, lab_name, lab_ref

#### 6. Validate extracted values
Checks that values fall within medically plausible ranges.

In [8]:
# (min, max) plausible ranges — outside these = data quality issue
RANGES = {
    "glucose":           (20,   600),
    "hemoglobin":        (3,    25),
    "cholesterol_total": (50,   500),
    "cholesterol_hdl":   (10,   150),
    "cholesterol_ldl":   (10,   400),
    "triglycerides":     (20,   2000),
    "wbc":               (0.5,  100),
    "rbc":               (1,    10),
}

def validate(markers):
    """Return (is_valid, notes). Flags values outside plausible ranges."""
    issues = []
    for field, (lo, hi) in RANGES.items():
        val = markers.get(field)
        if val is not None and not (lo <= val <= hi):
            issues.append(f"{field}={val} out of range [{lo}–{hi}]")
    return (len(issues) == 0), "; ".join(issues) or None

#### 7. Save one report to PostgreSQL
`ON CONFLICT DO NOTHING` ensures **no duplicate records** are ever inserted.

In [9]:
INSERT_SQL = """
INSERT INTO blood_reports (
    customer_id, report_date, lab_name, lab_ref,
    glucose, hemoglobin, cholesterol_total, cholesterol_hdl,
    cholesterol_ldl, triglycerides, wbc, rbc,
    raw_data, minio_path, extraction_method, is_valid, validation_notes
) VALUES (
    %(customer_id)s, %(report_date)s, %(lab_name)s, %(lab_ref)s,
    %(glucose)s, %(hemoglobin)s, %(cholesterol_total)s, %(cholesterol_hdl)s,
    %(cholesterol_ldl)s, %(triglycerides)s, %(wbc)s, %(rbc)s,
    %(raw_data)s, %(minio_path)s, %(extraction_method)s,
    %(is_valid)s, %(validation_notes)s
)
ON CONFLICT (customer_id, report_date, lab_name, lab_ref) DO NOTHING
"""

def save_report(pg, record):
    with pg.cursor() as cur:
        cur.execute(INSERT_SQL, record)
        inserted = cur.rowcount == 1
    pg.commit()
    return inserted



#### 8. Process one PDF end-to-end

In [19]:
BUCKET = "health-data"

def process_pdf(minio, pg, object_path, customer_id):
    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    tmp_path = tmp.name
    tmp.close()

    try:
        minio.fget_object(BUCKET, object_path, tmp_path)

        text, method     = extract_text(tmp_path)
        markers          = parse_markers(text)

        report_date, lab, extracted_ref = parse_meta(text)
        is_valid, notes  = validate(markers)

        record = {
            "customer_id":       customer_id,
            "report_date":       report_date,
            "lab_name":          lab,
            "lab_ref":           object_path,   # ← unique file path as lab_ref
            **markers,
            "raw_data":          json.dumps(markers),
            "minio_path":        object_path,
            "extraction_method": method,
            "is_valid":          is_valid,
            "validation_notes":  notes,
        }

        inserted = save_report(pg, record)
        status   = "✅ Saved" if inserted else "⏭️  Duplicate — skipped"
        print(f"  {status}: {object_path}")

    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)



#### 9. Sync all blood reports from MinIO

Scans the entire bucket for PDFs under `blood-reports/` folders.
- ✅ New customer added to MinIO → picked up automatically
- ✅ New PDF added to existing customer → picked up automatically  
- ✅ Re-run anytime — duplicates are silently skipped

In [24]:
def sync_all_blood_reports(minio, pg):
    """Scan MinIO for all blood report PDFs and process new ones."""
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    pdfs    = [
        obj for obj in objects
        if "blood-reports" in obj.object_name
        and obj.object_name.endswith(".pdf")
    ]

    print(f"Found {len(pdfs)} blood report PDF(s) in MinIO\n")

    for obj in pdfs:
        parts       = obj.object_name.split("/")
        customer_id = parts[1]            # customers/{customer_id}/blood-reports/file.pdf
        print(f"👤 {customer_id}")
        process_pdf(minio, pg, obj.object_name, customer_id)

    print("\n🎉 Sync complete")


# ▶️ Run the full sync
sync_all_blood_reports(minio, pg)

Found 5 blood report PDF(s) in MinIO

👤 CUST_amara_patel_03630F04
  ⏭️  Duplicate — skipped: customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf
👤 CUST_carlos_rivera_5A81755B
  ⏭️  Duplicate — skipped: customers/CUST_carlos_rivera_5A81755B/blood-reports/BloodReport_CUST_car_20260511.pdf
👤 CUST_fatima_alsayed_8594AA83
  ⏭️  Duplicate — skipped: customers/CUST_fatima_alsayed_8594AA83/blood-reports/BloodReport_CUST_fat_20260511.pdf
👤 CUST_john_whitfield_C4987FD5
  ⏭️  Duplicate — skipped: customers/CUST_john_whitfield_C4987FD5/blood-reports/BloodReport_CUST_joh_20260511.pdf
👤 CUST_mei_lin_66CBFE0F
  ⏭️  Duplicate — skipped: customers/CUST_mei_lin_66CBFE0F/blood-reports/BloodReport_CUST_mei_20260511.pdf

🎉 Sync complete


#### 10.  Auto-sync on a schedule (optional)

In [ ]:
!pip install apscheduler --quiet

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
from datetime import datetime, timezone

def scheduled_sync():
    print(f"\n⏰ Auto-sync at {datetime.now(timezone.utc).strftime('%H:%M UTC')}")
    sync_all_blood_reports(get_minio(), get_pg())

scheduler = BackgroundScheduler()
scheduler.add_job(scheduled_sync, "interval", minutes=30)  # ✏️ adjust interval
scheduler.start()
print("⏰ Scheduler started — syncing every 30 min. Run scheduler.shutdown() to stop.")

In [ ]:
# ⛔ Stop the scheduler
scheduler.shutdown()
print("Scheduler stopped.")

#### 11.  Verify — query the results

In [ ]:
def query(pg, sql, label):
    with pg.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(f"\n📊 {label}")
    print("  " + " | ".join(cols))
    print("  " + "-" * 70)
    for row in rows:
        print("  " + " | ".join(str(v) for v in row))

# All reports
query(pg,
    "SELECT customer_id, report_date, lab_name, glucose, hemoglobin, is_valid FROM blood_reports ORDER BY customer_id",
    "All blood reports"
)

# Reports with validation issues
query(pg,
    "SELECT customer_id, report_date, validation_notes FROM blood_reports WHERE is_valid = FALSE",
    "Reports with validation issues"
)

# Count per customer
query(pg,
    "SELECT customer_id, COUNT(*) as reports FROM blood_reports GROUP BY customer_id",
    "Reports per customer"
)